In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_scheduler
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
from tqdm import tqdm
import random

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Dataset Class
class SexismDataset(Dataset):
    def __init__(self, tweets, labels, tokenizer, max_len):
        self.tweets = tweets
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.tweets)

    def __getitem__(self, idx):
        tweet = str(self.tweets[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(tweet, 
                                   max_length=self.max_len, 
                                   padding='max_length', 
                                   truncation=True, 
                                   return_tensors='pt')

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# DataLoader Function
def create_data_loader(df, tokenizer, max_len, batch_size):
    dataset = SexismDataset(
        tweets=df['tweet'].to_numpy(),
        labels=df['label'].to_numpy(),
        tokenizer=tokenizer,
        max_len=max_len
    )
    return DataLoader(dataset, batch_size=batch_size, num_workers=4)

# Training Function
def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    total_loss, correct_predictions = 0, 0

    for batch in tqdm(data_loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return correct_predictions.double() / len(data_loader.dataset), total_loss / len(data_loader)

# Evaluation Function
def eval_model(model, data_loader, device):
    model.eval()
    total_loss, correct_predictions = 0, 0
    preds_list, labels_list = [], []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            preds = torch.argmax(logits, dim=1)
            correct_predictions += torch.sum(preds == labels)
            total_loss += loss.item()

            preds_list.extend(preds.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())

    return correct_predictions.double() / len(data_loader.dataset), total_loss / len(data_loader), preds_list, labels_list

# Baseline Models Function
def train_baseline_models(train_df, val_df):
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

    X_train = vectorizer.fit_transform(train_df['tweet'])
    y_train = train_df['label']
    X_val = vectorizer.transform(val_df['tweet'])
    y_val = val_df['label']

    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000),
        'Naive Bayes': MultinomialNB(),
        'Decision Tree': DecisionTreeClassifier()
    }

    for model_name, model in models.items():
        print(f"Training {model_name}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        print(f"Performance for {model_name}:")
        print(classification_report(y_val, y_pred, target_names=['NO', 'YES']))

# Main Workflow
data_path = 'EXIST2024_training_Task1.csv'
df = pd.read_csv(data_path, encoding='utf-8')

# Process Labels
df['labels_list'] = df['labels_task1'].apply(eval)
df['majority_label'] = df['labels_list'].apply(lambda x: Counter(x).most_common(1)[0][0])
df['label'] = df['majority_label'].map({'YES': 1, 'NO': 0})
df = df[df['label'].notnull()]

# Hyperparameters
MAX_LEN = 512
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 3e-5

# Process by Language
for lang in ['en', 'es']:
    lang_df = df[df['lang'] == lang]
    train_df, test_df = train_test_split(lang_df, test_size=0.1, random_state=SEED, stratify=lang_df['label'])

    print(f"\nBaseline Models for {lang.upper()}:")
    train_baseline_models(train_df, test_df)

    # Initialize Tokenizer and Model
    tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
    model = BertForSequenceClassification.from_pretrained('bert-base-multilingual-cased', num_labels=2).to(device)

    # Data Loaders
    train_loader = create_data_loader(train_df, tokenizer, MAX_LEN, BATCH_SIZE)
    test_loader = create_data_loader(test_df, tokenizer, MAX_LEN, BATCH_SIZE)

    # Optimizer and Scheduler
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    print(f"\nTraining BERT for {lang.upper()}:")
    best_accuracy = 0
    for epoch in range(EPOCHS):
        train_acc, train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}")

        val_acc, val_loss, preds, labels = eval_model(model, test_loader, device)
        print(f"Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}")

        if val_acc > best_accuracy:
            torch.save(model.state_dict(), f'best_model_{lang}.bin')
            best_accuracy = val_acc

    print(f"\nFinal Evaluation for {lang.upper()}:")
    model.load_state_dict(torch.load(f'best_model_{lang}.bin'))
    test_acc, test_loss, preds, labels = eval_model(model, test_loader, device)
    print(classification_report(labels, preds, target_names=['NO', 'YES']))

/home/hmohammadi/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Baseline Models for EN:
Training Logistic Regression...
Performance for Logistic Regression:
              precision    recall  f1-score   support

          NO       0.74      0.89      0.81       194
         YES       0.76      0.54      0.63       132

    accuracy                           0.75       326
   macro avg       0.75      0.71      0.72       326
weighted avg       0.75      0.75      0.73       326

Training Naive Bayes...
Performance for Naive Bayes:
              precision    recall  f1-score   support

          NO       0.74      0.93      0.82       194
         YES       0.84      0.51      0.63       132

    accuracy                           0.76       326
   macro avg       0.79      0.72      0.73       326
weighted avg       0.78      0.76      0.75       326

Training Decision Tree...
Performance for Decision Tree:
              precision    recall  f1-score   support

          NO       0.77      0.74      0.76       194
         YES       0.64      0.68

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/hmohammadi/.local/lib/python3.10/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(



Training BERT for EN:


Training: 100%|██████████| 184/184 [01:15<00:00,  2.43it/s]


...

Val Loss: 0.6582, Accuracy: 0.7705


/tmp/ipykernel_816113/1663224054.py:189: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'best_model_{lang}.bin'))



Final Evaluation for ES:


Evaluating: 100%|██████████| 23/23 [00:03<00:00,  7.32it/s]

              precision    recall  f1-score   support

          NO       0.80      0.74      0.77       188
         YES       0.75      0.80      0.77       178

    accuracy                           0.77       366
   macro avg       0.77      0.77      0.77       366
weighted avg       0.77      0.77      0.77       366



In [2]:
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_scheduler
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
from tqdm import tqdm
import random

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Dataset Class
class SexismDataset(Dataset):
    def __init__(self, tweets, labels, tokenizer, max_len):
        self.tweets = tweets
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.tweets)

    def __getitem__(self, idx):
        tweet = str(self.tweets[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(tweet, 
                                   max_length=self.max_len, 
                                   padding='max_length', 
                                   truncation=True, 
                                   return_tensors='pt')

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# DataLoader Function
def create_data_loader(df, tokenizer, max_len, batch_size):
    dataset = SexismDataset(
        tweets=df['tweet'].to_numpy(),
        labels=df['label'].to_numpy(),
        tokenizer=tokenizer,
        max_len=max_len
    )
    return DataLoader(dataset, batch_size=batch_size, num_workers=4)

# Training Function
def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    total_loss, correct_predictions = 0, 0

    for batch in tqdm(data_loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return correct_predictions.double() / len(data_loader.dataset), total_loss / len(data_loader)

# Evaluation Function
def eval_model(model, data_loader, device):
    model.eval()
    total_loss, correct_predictions = 0, 0
    preds_list, labels_list = [], []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            preds = torch.argmax(logits, dim=1)
            correct_predictions += torch.sum(preds == labels)
            total_loss += loss.item()

            preds_list.extend(preds.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())

    return correct_predictions.double() / len(data_loader.dataset), total_loss / len(data_loader), preds_list, labels_list

# Baseline Models Function
def train_baseline_models(train_df, val_df):
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

    X_train = vectorizer.fit_transform(train_df['tweet'])
    y_train = train_df['label']
    X_val = vectorizer.transform(val_df['tweet'])
    y_val = val_df['label']

    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000),
        'Naive Bayes': MultinomialNB(),
        'Decision Tree': DecisionTreeClassifier()
    }

    for model_name, model in models.items():
        print(f"Training {model_name}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        print(f"Performance for {model_name}:")
        print(classification_report(y_val, y_pred, target_names=['NO', 'YES']))

# Main Workflow
data_path = 'EXIST2024_training_Task1.csv'
df = pd.read_csv(data_path, encoding='utf-8')

# Process Labels
df['labels_list'] = df['labels_task1'].apply(eval)
df['majority_label'] = df['labels_list'].apply(lambda x: Counter(x).most_common(1)[0][0])
df['label'] = df['majority_label'].map({'YES': 1, 'NO': 0})
df = df[df['label'].notnull()]

# Hyperparameters
MAX_LEN = 512
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 3e-5

# Process by Language
for lang in ['en', 'es']:
    lang_df = df[df['lang'] == lang]
    train_df, test_df = train_test_split(lang_df, test_size=0.1, random_state=SEED, stratify=lang_df['label'])

    print(f"\nLanguage: {lang.upper()}")
    print(f"Training samples: {len(train_df)}")
    print(f"Testing samples: {len(test_df)}")

    print(f"\nBaseline Models for {lang.upper()}:")
    train_baseline_models(train_df, test_df)

    # Initialize Tokenizer and Model
    tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
    model = BertForSequenceClassification.from_pretrained('bert-base-multilingual-cased', num_labels=2).to(device)

    # Data Loaders
    train_loader = create_data_loader(train_df, tokenizer, MAX_LEN, BATCH_SIZE)
    test_loader = create_data_loader(test_df, tokenizer, MAX_LEN, BATCH_SIZE)

    # Optimizer and Scheduler
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    print(f"\nTraining BERT for {lang.upper()}:")
    best_accuracy = 0
    for epoch in range(EPOCHS):
        train_acc, train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}")

        val_acc, val_loss, preds, labels = eval_model(model, test_loader, device)
        print(f"Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}")

        if val_acc > best_accuracy:
            torch.save(model.state_dict(), f'best_model_{lang}.bin')
            best_accuracy = val_acc

    print(f"\nFinal Evaluation for {lang.upper()}:")
    model.load_state_dict(torch.load(f'best_model_{lang}.bin'))
    test_acc, test_loss, preds, labels = eval_model(model, test_loader, device)
    print(classification_report(labels, preds, target_names=['NO', 'YES']))


Language: EN
Training samples: 2934
Testing samples: 326

Baseline Models for EN:
Training Logistic Regression...
Performance for Logistic Regression:
              precision    recall  f1-score   support

          NO       0.74      0.89      0.81       194
         YES       0.76      0.54      0.63       132

    accuracy                           0.75       326
   macro avg       0.75      0.71      0.72       326
weighted avg       0.75      0.75      0.73       326

Training Naive Bayes...
Performance for Naive Bayes:
              precision    recall  f1-score   support

          NO       0.74      0.93      0.82       194
         YES       0.84      0.51      0.63       132

    accuracy                           0.76       326
   macro avg       0.79      0.72      0.73       326
weighted avg       0.78      0.76      0.75       326

Training Decision Tree...
Performance for Decision Tree:
              precision    recall  f1-score   support

          NO       0.77      

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/hmohammadi/.local/lib/python3.10/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(



Training BERT for EN:


Training: 100%|██████████| 23/23 [01:06<00:00,  2.88s/it]


Epoch 1 | Train Loss: 0.6774, Accuracy: 0.5903


...

Epoch 10 | Train Loss: 0.0686, Accuracy: 0.9791


Evaluating: 100%|██████████| 3/3 [00:02<00:00,  1.03it/s]
/tmp/ipykernel_816113/2842585484.py:193: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.

Val Loss: 0.8266, Accuracy: 0.7596

Final Evaluation for ES:


Evaluating: 100%|██████████| 3/3 [00:02<00:00,  1.03it/s]

              precision    recall  f1-score   support

          NO       0.77      0.81      0.79       188
         YES       0.79      0.75      0.77       178

    accuracy                           0.78       366
   macro avg       0.78      0.78      0.78       366
weighted avg       0.78      0.78      0.78       366

